# Prediction of TMPRSS2–ERG status

This notebook is an additional machine-learning experiment based on the earlier prostate-cancer cohort analysis. The target is **TMPRSS2–ERG structural-variant status** in TCGA. The predictors are a fixed 92-gene expression panel plus age and pathological T/N stage.

The evaluation is internal to TCGA and should not be read as a clinical diagnostic result.

## 1. Design before modelling

- Split patients once into 80% development and 20% untouched test data.
- Keep all preprocessing, feature selection, tuning and threshold selection inside the development data.
- Exclude `ERG`, `TMPRSS2` and the target-derived clinical `SUBTYPE` field.
- Compare a clinical baseline, a regularised expression baseline, an expression MLP and a multimodal PyTorch MLP.
- Evaluate discrimination, threshold-dependent performance, probability calibration and uncertainty.

A 50/50 split would leave only about 246 patients for model development. The 80/20 split preserves 394 development patients while still retaining a 99-patient independent test set.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import SVG, display

from modeling import run_experiment

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / 'data' / 'model_input.csv'
RESULTS_DIR = PROJECT_DIR / 'results'
print(DATA_PATH)

## 2. Audit the prepared public data

The repository includes the prepared modelling table so this notebook can run without contacting the API. `data_pipeline.py` reproduces it from cBioPortal; `data_manifest.json` records the exact profiles, transform, counts and missingness.

In [ ]:
data = pd.read_csv(DATA_PATH)
manifest = json.loads((PROJECT_DIR / 'data' / 'data_manifest.json').read_text())

audit = pd.Series({
    'patients': data['patient_id'].nunique(),
    'samples': len(data),
    'fusion positive': int(data['fusion_status'].sum()),
    'fusion negative': int((1 - data['fusion_status']).sum()),
    'expression predictors': sum(column.startswith('expr__') for column in data),
    'clinical predictors': sum(column.startswith('clinical__') for column in data),
})
display(audit.to_frame('value'))
assert 'expr__ERG' not in data and 'expr__TMPRSS2' not in data
assert data['patient_id'].nunique() == len(data)
display(data.head())

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame('missing values'))

## 3. Train and evaluate

The next cell runs the models. Full PyTorch cross-validation takes longer than the linear baselines on a CPU. `QUICK_RUN = True` is only meant for checking the installation; the reported results use the full settings.

In [ ]:
QUICK_RUN = False

run_experiment(
    data_path=DATA_PATH,
    output_dir=RESULTS_DIR,
    test_fraction=0.20,
    cv_folds=5,
    seed=2026,
    bootstrap_repeats=100 if QUICK_RUN else 1000,
    max_epochs=50 if QUICK_RUN else 300,
    patience=10 if QUICK_RUN else 25,
)

## 4. Compare held-out test performance

ROC-AUC and PR-AUC evaluate ranking across thresholds. Balanced accuracy gives sensitivity and specificity equal weight. Brier score evaluates probability error and is better when smaller. Sensitivity, specificity, precision and F1 use a threshold selected from out-of-fold development predictions—not from the test set.

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / 'metrics.csv')
display(metrics.set_index('model'))
display(SVG(filename=str(RESULTS_DIR / 'roc_curves.svg')))
display(SVG(filename=str(RESULTS_DIR / 'calibration.svg')))

## 5. Inspect model explanations

For logistic regression, importance is the signed coefficient after standardisation. For the PyTorch models, importance is the mean absolute Integrated Gradient on held-out samples. These values describe the fitted model; they do not establish causal biology.

In [ ]:
top_features = pd.read_csv(RESULTS_DIR / 'top_features.csv')
for model, table in top_features.groupby('model', sort=False):
    print(f'\n{model}')
    display(table.head(10).set_index('rank'))

## 6. Interpretation

The strongest model should be chosen from the **untouched test metrics together with calibration and uncertainty**, not from training accuracy. If the elastic net matches or beats the MLP, that is a scientifically useful result: with roughly 400 development patients, a regularised linear decision boundary can be more stable than a neural network. If the multimodal MLP improves on both the clinical and expression-only models, the clinical branch adds complementary information.

Even a strong TCGA test result only supports internal generalisation to patients measured like TCGA. A later SU2C analysis should be labelled as a domain-shift stress test because its stage distribution and laboratory workflow differ.